# Task 3 – Running & Benchmarking a Local LLM

This notebook covers:
1. **Setup** – installing Ollama and pulling models
2. **Benchmark** – running 10 standardized prompts on two quantisation levels
3. **Results** – speed (tok/s), memory (VRAM), and quality comparison tables
4. **Written answers** – KV cache explanation, quantisation memory savings, 24 GB serving plan

> **Prerequisites**: Ollama installed and running (`ollama serve`) on `http://localhost:11434`

## 1. Environment Setup

In [ ]:
# Install dependencies (run once)
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'requests', 'psutil', 'tabulate', 'nvidia-ml-py3'])
print('✔ Dependencies installed')

In [ ]:
# Verify Ollama is reachable
import requests

OLLAMA_BASE = 'http://localhost:11434'

try:
    r = requests.get(f'{OLLAMA_BASE}/api/tags', timeout=5)
    models = [m['name'] for m in r.json().get('models', [])]
    print(f'✔ Ollama running. Local models: {models if models else "(none yet)"}')
except Exception as e:
    print(f'✘ Cannot reach Ollama: {e}')
    print('  Start it with: ollama serve')

### Pull the benchmark models

We benchmark two quantisation variants of Llama 3.2 1B:

| Model tag | Quantisation | Approx size |
|---|---|---|
| `llama3.2:1b` | Q4_K_M (default) | ~0.75 GB |
| `llama3.2:1b:q4_0` | Q4_0 (older 4-bit) | ~0.70 GB |

In [ ]:
MODELS = ['llama3.2:1b', 'llama3.2:1b:q4_0']

def pull_model(model: str) -> bool:
    print(f'  Pulling {model} …')
    try:
        with requests.post(
            f'{OLLAMA_BASE}/api/pull',
            json={'name': model, 'stream': True},
            stream=True, timeout=600
        ) as resp:
            resp.raise_for_status()
            last = ''
            for line in resp.iter_lines():
                if line:
                    data = __import__('json').loads(line)
                    s = data.get('status', '')
                    if s != last:
                        print(f'    [{s}]')
                        last = s
            print(f'  ✔ {model} ready')
            return True
    except Exception as exc:
        print(f'  ✘ {exc}')
        return False

try:
    local = [m['name'] for m in requests.get(f'{OLLAMA_BASE}/api/tags', timeout=5).json().get('models', [])]
    for m in MODELS:
        base = m.split(':')[0]
        if any(base in l for l in local):
            print(f'✔ {m} already present')
        else:
            pull_model(m)
except Exception as e:
    print(f'Could not check/pull models: {e}')

## 2. The 10 Test Prompts

In [ ]:
TEST_PROMPTS = [
    # Factual Q&A
    {'id': 1, 'category': 'factual_qa',
     'prompt': 'What is the speed of light in a vacuum, and what are its implications for space travel to the nearest star system?'},
    {'id': 2, 'category': 'factual_qa',
     'prompt': 'Explain the difference between RAM and ROM in a computer system. Give two real-world examples of each.'},
    # Code generation
    {'id': 3, 'category': 'code_generation',
     'prompt': 'Write a Python function that takes a list of integers and returns the two numbers that sum to a given target value. Include a docstring and usage example.'},
    {'id': 4, 'category': 'code_generation',
     'prompt': 'Write a SQL query that finds the top 5 customers by total purchase amount from a table called `orders` with columns: customer_id, amount, order_date.'},
    # Summarization
    {'id': 5, 'category': 'summarization',
     'prompt': 'Summarize the main ideas of machine learning in three concise bullet points suitable for a non-technical audience.'},
    {'id': 6, 'category': 'summarization',
     'prompt': 'Summarize the pros and cons of renewable energy vs fossil fuels in under 100 words.'},
    # Reasoning
    {'id': 7, 'category': 'reasoning',
     'prompt': 'If a train leaves Station A at 9 AM at 80 km/h and another leaves Station B (320 km away) at 10 AM at 100 km/h toward Station A, at what time do they meet? Show working.'},
    {'id': 8, 'category': 'reasoning',
     'prompt': 'You have 3 boxes: one has only apples, one only oranges, one both. All labels are wrong. Pick one fruit from one box. How do you correctly label all boxes? Explain step by step.'},
    # Creative writing
    {'id': 9, 'category': 'creative_writing',
     'prompt': 'Write a short (4-sentence) science-fiction story opening set on a generation ship that has just lost contact with Earth.'},
    {'id': 10, 'category': 'creative_writing',
     'prompt': 'Write a haiku about artificial intelligence discovering emotions for the first time.'},
]

# Display the prompts nicely
for p in TEST_PROMPTS:
    print(f"[{p['id']:02d}] [{p['category']}]")
    print(f"  {p['prompt']}")
    print()

## 3. Benchmark Engine

In [ ]:
import json, time, subprocess
from datetime import datetime, timezone
import psutil

NS_PER_SEC = 1_000_000_000
GENERATE_EP = f'{OLLAMA_BASE}/api/generate'
PS_EP        = f'{OLLAMA_BASE}/api/ps'


def snapshot_memory(model: str) -> dict:
    """Get VRAM usage from Ollama /api/ps, with nvidia-smi fallback."""
    mem = {}
    # Ollama /api/ps
    try:
        ps = requests.get(PS_EP, timeout=5).json().get('models', [])
        for m in ps:
            base = model.split(':')[0]
            if base in m.get('name', ''):
                vram = m.get('size_vram', m.get('size', 0))
                mem['vram_mb'] = round(vram / 1024**2, 1)
                mem['source']  = 'ollama_ps'
                break
        if ps and 'vram_mb' not in mem:
            # fall back to first loaded model
            vram = ps[0].get('size_vram', ps[0].get('size', 0))
            mem['vram_mb'] = round(vram / 1024**2, 1)
            mem['source']  = 'ollama_ps_first'
    except Exception:
        pass

    if 'vram_mb' not in mem:
        # nvidia-smi fallback
        try:
            out = subprocess.check_output(
                ['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader,nounits'],
                timeout=5, text=True)
            mem['vram_mb'] = int(out.strip().splitlines()[0])
            mem['source']  = 'nvidia_smi'
        except Exception:
            mem['vram_mb'] = None
            mem['source']  = 'unavailable'

    mem['rss_mb'] = round(psutil.Process().memory_info().rss / 1024**2, 1)
    return mem


def run_prompt(model: str, prompt_obj: dict, timeout: int = 120) -> dict:
    """Run one prompt and return metrics."""
    t0 = time.perf_counter()
    try:
        r = requests.post(
            GENERATE_EP,
            json={'model': model, 'prompt': prompt_obj['prompt'],
                  'stream': False,
                  'options': {'temperature': 0.1, 'num_predict': 256}},
            timeout=timeout,
        )
        r.raise_for_status()
        t1 = time.perf_counter()
        d  = r.json()
        ec = d.get('eval_count', 0)
        ed = d.get('eval_duration', 1)
        tps = ec / (ed / NS_PER_SEC) if ec and ed else 0.0
        return {
            'prompt_id'         : prompt_obj['id'],
            'category'          : prompt_obj['category'],
            'prompt'            : prompt_obj['prompt'],
            'response'          : d.get('response', ''),
            'tokens_generated'  : ec,
            'tokens_per_second' : round(tps, 2),
            'eval_duration_s'   : round(ed / NS_PER_SEC, 3),
            'wall_clock_s'      : round(t1 - t0, 3),
            'error'             : None,
        }
    except Exception as exc:
        return {
            'prompt_id': prompt_obj['id'], 'category': prompt_obj['category'],
            'prompt': prompt_obj['prompt'], 'response': '',
            'tokens_generated': 0, 'tokens_per_second': 0.0,
            'eval_duration_s': 0.0, 'wall_clock_s': round(time.perf_counter()-t0, 3),
            'error': str(exc),
        }


def benchmark_model(model: str, prompts: list, timeout: int = 120) -> dict:
    print(f'\n{"="*55}')
    print(f'  Benchmarking: {model}')
    print(f'{"="*55}')
    # warmup
    print('  Warmup …')
    run_prompt(model, {'id':0, 'category':'warmup', 'prompt':'Say hello.'}, timeout)
    mem = snapshot_memory(model)
    print(f'  VRAM: {mem["vram_mb"]} MB ({mem["source"]})')

    results = []
    for p in prompts:
        print(f'  [{p["id"]:02d}/10] {p["category"]:18s} ', end='', flush=True)
        res = run_prompt(model, p, timeout)
        results.append(res)
        if res['error']:
            print(f'ERROR – {res["error"]}')
        else:
            print(f'{res["tokens_per_second"]:6.1f} tok/s  {res["tokens_generated"]:4d} tokens  {res["wall_clock_s"]:6.2f}s')

    valid  = [r for r in results if not r['error'] and r['tokens_generated'] > 0]
    avg_tps = sum(r['tokens_per_second'] for r in valid) / len(valid) if valid else 0
    return {
        'model': model,
        'timestamp': datetime.now(timezone.utc).isoformat(),
        'avg_tokens_per_sec': round(avg_tps, 2),
        'avg_wall_clock_s'  : round(sum(r['wall_clock_s'] for r in valid)/len(valid), 3) if valid else 0,
        'total_tokens_generated': sum(r['tokens_generated'] for r in valid),
        'successful_prompts': len(valid),
        'errors': len(results)-len(valid),
        'memory': mem,
        'prompt_results': results,
    }

print('✔ Benchmark engine ready')

## 4. Run the Benchmark

> **Note**: Each model processes 10 prompts with up to 256 output tokens.  
> Expect ~2–10 minutes depending on your hardware.

In [ ]:
all_results = []

for model in MODELS:
    result = benchmark_model(model, TEST_PROMPTS, timeout=180)
    all_results.append(result)

# Save to JSON
output = {
    'benchmark_meta': {
        'run_at': datetime.now(timezone.utc).isoformat(),
        'models': MODELS,
        'num_prompts': len(TEST_PROMPTS),
    },
    'results': all_results,
}
with open('benchmark_results.json', 'w') as f:
    json.dump(output, f, indent=2)

print('\n✔ Results saved to benchmark_results.json')

## 5. Results Tables

In [ ]:
from tabulate import tabulate

# ── Summary table ─────────────────────────────────────────────────────────────
summary_rows = []
for r in all_results:
    vram = r['memory'].get('vram_mb')
    summary_rows.append([
        r['model'],
        f"{r['avg_tokens_per_sec']:.1f}",
        f"{vram} MB" if vram else 'N/A',
        f"{r['avg_wall_clock_s']:.2f}s",
        r['successful_prompts'],
        r['errors'],
    ])

print(tabulate(
    summary_rows,
    headers=['Model', 'Avg tok/s', 'VRAM Used', 'Avg Latency', 'OK', 'Errors'],
    tablefmt='rounded_outline'
))

# ── Speed comparison ──────────────────────────────────────────────────────────
if len(all_results) == 2:
    tps1 = all_results[0]['avg_tokens_per_sec']
    tps2 = all_results[1]['avg_tokens_per_sec']
    if tps1 and tps2:
        faster = all_results[0] if tps1 > tps2 else all_results[1]
        ratio  = max(tps1, tps2) / min(tps1, tps2)
        print(f'\n🏆 Speed winner: {faster["model"]}  ({ratio:.2f}× faster)')

    v1 = all_results[0]['memory'].get('vram_mb')
    v2 = all_results[1]['memory'].get('vram_mb')
    if v1 and v2:
        saving = abs(v1-v2)/max(v1,v2)*100
        smaller = all_results[0] if v1 < v2 else all_results[1]
        print(f'💾 VRAM saving : {saving:.1f}% less  ({smaller["model"]})')

In [ ]:
# ── Per-prompt breakdown ──────────────────────────────────────────────────────
if len(all_results) == 2:
    m1, m2 = all_results
    rows = []
    for p1, p2 in zip(m1['prompt_results'], m2['prompt_results']):
        rows.append([
            p1['prompt_id'],
            p1['category'],
            f"{p1['tokens_per_second']:.1f}" if not p1['error'] else 'ERR',
            f"{p2['tokens_per_second']:.1f}" if not p2['error'] else 'ERR',
            p1['tokens_generated'] if not p1['error'] else '–',
            p2['tokens_generated'] if not p2['error'] else '–',
        ])
    print(tabulate(
        rows,
        headers=['#', 'Category',
                 f'tok/s\n{m1["model"]}', f'tok/s\n{m2["model"]}',
                 f'tokens\n{m1["model"]}', f'tokens\n{m2["model"]}'],
        tablefmt='rounded_outline'
    ))

## 6. Output Quality Comparison

In [ ]:
import re

def score_response(pr: dict) -> dict:
    text  = pr.get('response', '') or ''
    error = pr.get('error')
    cat   = pr.get('category', '')
    if error or not text:
        return {'total': 0.0, 'word_count': 0}
    words = len(text.split())
    # Length
    length_score = 10 if words >= 100 else 8 if words >= 50 else 6 if words >= 25 else 4 if words >= 10 else 2
    # Structure
    structure_score = min(10.0, 2.5 * sum([
        bool(re.search(r'^[\-\*•]\s', text, re.MULTILINE)),
        bool(re.search(r'^\d+[\.\.)]\s', text, re.MULTILINE)),
        '```' in text,
        bool(re.search(r'^#{1,3}\s', text, re.MULTILINE)),
    ]))
    # Reasoning
    kws = ['therefore', 'because', 'step', 'km/h', '='] if cat == 'reasoning' else ['def ', 'SELECT', 'return']
    reasoning_score = min(10.0, sum(1 for kw in kws if kw.lower() in text.lower()) * 2.0)
    total = (length_score + structure_score + reasoning_score + 10) / 4
    return {'total': round(total, 2), 'word_count': words}

if len(all_results) == 2:
    qual_rows = []
    for idx in range(len(all_results[0]['prompt_results'])):
        pr1 = all_results[0]['prompt_results'][idx]
        pr2 = all_results[1]['prompt_results'][idx]
        s1  = score_response(pr1)
        s2  = score_response(pr2)
        qual_rows.append([
            pr1['prompt_id'],
            pr1['category'],
            f"{s1['total']:.1f} ({s1['word_count']}w)",
            f"{s2['total']:.1f} ({s2['word_count']}w)",
        ])
    print(tabulate(
        qual_rows,
        headers=['#', 'Category', f"Quality\n{all_results[0]['model']}", f"Quality\n{all_results[1]['model']}"],
        tablefmt='rounded_outline',
    ))
    for r in all_results:
        avg_q = sum(score_response(pr)['total'] for pr in r['prompt_results']) / len(r['prompt_results'])
        print(f"Avg quality  {r['model']:35s}: {avg_q:.2f} / 10")

In [ ]:
# ── Side-by-side response viewer ─────────────────────────────────────────────
PROMPT_ID_TO_VIEW = 7   # change this to view any prompt 1–10

if len(all_results) >= 2:
    idx = PROMPT_ID_TO_VIEW - 1
    for r in all_results:
        pr = r['prompt_results'][idx]
        print(f"\n{'─'*55}")
        print(f"Model: {r['model']}")
        print(f"{'─'*55}")
        print(pr['response'][:800] + ('…' if len(pr.get('response','')) > 800 else ''))

## 7. Written Answers

### 7a. What Is the KV Cache?

In a Transformer decoder, each **attention layer** computes:

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

During **autoregressive generation** (one token at a time), the K and V matrices for all previous tokens would need to be **recomputed from scratch** at every step — an $O(n^2)$ cost.

The **KV cache** stores those computed K and V tensors. At each step, only the *new* token's K/V pair is computed and appended to the cache; the attention operation reads the full history from cache.  
This reduces decode-step cost from $O(n^2)$ to $O(n)$ and enables real-time generation speeds.

**Memory formula:**

$$\text{KV\_bytes} = 2 \times L \times S \times d_{\text{model}} \times B_{\text{elem}}$$

where $L$ = layers, $S$ = sequence length, $d_{\text{model}}$ = hidden dimension, $B_{\text{elem}}$ = bytes per element (2 for fp16).

**Example** — Llama-3.2-1B, 2 048 context, fp16:
$$= 2 \times 16 \times 2048 \times 2048 \times 2 \approx 0.25 \text{ GB per session}$$

---

### 7b. Why Does Quantisation Save Memory?

| Precision | Bits/param | 1B-param model |
|---|---|---|
| fp32 | 32 | 4 GB |
| fp16 / bf16 | 16 | 2 GB |
| Q8_0 | 8 | 1 GB |
| Q4_K_M | ~4.5 | **~0.75 GB** |
| Q4_0 | 4 | **~0.70 GB** |

Quantisation maps floating-point weights to lower-bit integers (+ per-block scale factors).  
During inference, weights are dequantised on the fly (a cheap operation) or computed with int8/int4 kernels.  
Memory savings: $1 - \frac{\text{bits}_{\text{quant}}}{\text{bits}_{\text{orig}}}$ → Q4 saves ~75% vs fp16.

**Trade-off**: minimal perplexity increase (~0.1–0.3 for Q4_K_M), faster decode, far less VRAM.

---

### 7c. 24 GB GPU Serving Plan — 20 Concurrent Users

**Assumptions**  
- GPU: 24 GB VRAM (RTX 3090 / A5000 / L4)  
- Model: Llama-3.2-7B at Q4_K_M  
- Context per user: 2 048 tokens  
- KV cache: fp16 (2 bytes/element)  
- Framework: Ollama with continuous batching  

**Memory budget:**

| Component | Calculation | Size |
|---|---|---|
| Model weights (7B, Q4_K_M) | 7B × 4.5 bits ÷ 8 ÷ 1024³ | ~3.9 GB |
| KV cache per user | 2×32×2048×4096×2 / 1024³ | ~0.5 GB |
| KV cache × 20 users | 20 × 0.5 GB | 10.0 GB |
| CUDA / framework | overhead | 0.5 GB |
| **Total** | | **14.4 GB ✅** |
| **Remaining VRAM** | 24 − 14.4 | **9.6 GB** |

**Recommended settings:**
```bash
OLLAMA_NUM_PARALLEL=8      # 8 active inference slots
OLLAMA_KEEP_ALIVE=10m      # evict idle models
OLLAMA_FLASH_ATTENTION=1   # reduce peak KV memory
```
Queue remaining users (average wait < 15 s at 30 tok/s generation speed).

In [ ]:
# ── 24 GB serving plan — interactive calculator ──────────────────────────────

def serving_plan(
    params_b:    float = 7.0,     # model params in billions
    bits:        float = 4.5,     # quantisation bits per param
    num_layers:  int   = 32,      # transformer layers
    d_model:     int   = 4096,    # hidden dimension
    ctx_tokens:  int   = 2048,    # max context per user
    kv_bytes:    int   = 2,       # bytes per KV element (2=fp16)
    num_users:   int   = 20,      # concurrent users
    gpu_vram_gb: float = 24.0,    # total VRAM
    overhead_gb: float = 0.5,     # framework overhead
):
    model_gb   = params_b * 1e9 * bits / 8 / 1024**3
    kv_per_user_gb = 2 * num_layers * ctx_tokens * d_model * kv_bytes / 1024**3
    kv_total_gb    = kv_per_user_gb * num_users
    total_gb       = model_gb + kv_total_gb + overhead_gb
    remaining_gb   = gpu_vram_gb - total_gb
    fits           = total_gb <= gpu_vram_gb

    print(f'Model:         {params_b}B params, {bits:.1f} bits/param')
    print(f'Layers/d_model: {num_layers} / {d_model}  |  Context: {ctx_tokens} tokens')
    print(f'  Model weights    : {model_gb:.2f} GB')
    print(f'  KV per user      : {kv_per_user_gb:.3f} GB')
    print(f'  KV x {num_users} users  : {kv_total_gb:.2f} GB')
    print(f'  Overhead         : {overhead_gb:.2f} GB')
    print(f'  ─────────────────────────')
    print(f'  TOTAL            : {total_gb:.2f} GB  {"✅ fits" if fits else "❌ exceeds"} {gpu_vram_gb} GB')
    print(f'  Remaining VRAM   : {remaining_gb:.2f} GB')
    return total_gb, fits

print('=== Scenario A: Llama-3.2-1B, Q4_K_M, 20 users, 2048 ctx ===')
serving_plan(params_b=1.24, bits=4.5, num_layers=16, d_model=2048, ctx_tokens=2048, num_users=20)

print()
print('=== Scenario B: Llama-3.2-7B, Q4_K_M, 20 users, 2048 ctx ===')
serving_plan(params_b=7.0,  bits=4.5, num_layers=32, d_model=4096, ctx_tokens=2048, num_users=20)

print()
print('=== Scenario C: Llama-3.2-7B, Q4_K_M, 20 users, 8192 ctx ===')
serving_plan(params_b=7.0,  bits=4.5, num_layers=32, d_model=4096, ctx_tokens=8192, num_users=20)

## 8. Summary

| Finding | Detail |
|---|---|
| **Q4_K_M vs Q4_0 speed** | Usually within 5–10% — quantisation format matters less than model size |
| **Memory saving** | ~72% fewer bytes vs fp16 at Q4_K_M with minimal quality loss |
| **KV cache bottleneck** | At high concurrency, KV cache dominates VRAM over model weights |
| **24 GB serving** | 7B model + 20 users at 2048 ctx = ~14 GB — fits comfortably |
| **Best practice** | Use Q4_K_M + Flash Attention 2 + continuous batching for production |

See `kv_cache_explainer.md` for the full technical writeup and all calculations.